# L6.2 — Few-Shot Learning and Chain-of-Thought

Hands-on notebook for the lesson [`6-2-few-shot-cot.mdx`](../../llm-quest-theory/level-6/6-2-few-shot-cot.mdx).

> **Learning objectives**
> - See in-context learning pick up a translation pattern with zero fine-tuning.
> - Run zero-shot vs few-shot vs **chain-of-thought** on a small word-problem suite and measure accuracy.
> - Build a minimal **self-consistency** sampler (majority vote over several CoT runs).
> - Show that CoT *wastes* tokens on trivially easy tasks — a sober counter-example.

## Connection to the theory
Covers **§1–§11** of the source `.mdx`. We use an instruction-tuned `flan-t5-base` (~250 M params) — small enough for CPU, big enough that CoT actually helps.

In [1]:
# ---- Setup ----
import os, re, warnings
from collections import Counter
warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42
torch.manual_seed(SEED)
DEVICE = "cpu"

MODEL_NAME = "google/flan-t5-base"    # ~250M params, still CPU-friendly
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()
print("model:", MODEL_NAME, "  params:", sum(p.numel() for p in model.parameters()))

model: google/flan-t5-base   params: 247577856


In [2]:
@torch.no_grad()
def generate(prompt, max_new_tokens=150, temperature=0.0, rng=None):
    ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).input_ids.to(DEVICE)
    if temperature <= 0:
        out = model.generate(ids, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
    else:
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=temperature, top_p=0.95)
    return tokenizer.decode(out[0], skip_special_tokens=True)

## 1. In-context learning — pattern transfer without fine-tuning
Translate via four English→French pairs, then ask for a fifth word the model never trained on.

In [3]:
shots = """Translate English to French.
sea otter => loutre de mer
peppermint => menthe poivrée
plush giraffe => girafe en peluche
cheese =>"""
print("prompt:"); print(shots); print("\nmodel :", generate(shots, max_new_tokens=10))

prompt:
Translate English to French.
sea otter => loutre de mer
peppermint => menthe poivrée
plush giraffe => girafe en peluche
cheese =>

model : otter => loutre de 


## 2. Zero-shot vs few-shot vs CoT on word problems
A small, hand-crafted set of five arithmetic word problems — the kind where CoT is supposed to shine.

In [4]:
PROBLEMS = [
    {
        "q": "A juggler has 16 balls. Half are golf balls, and half of the golf balls are blue. How many blue golf balls are there?",
        "a": 4,
    },
    {
        "q": "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 balls. How many tennis balls does he have now?",
        "a": 11,
    },
    {
        "q": "The cafeteria had 23 apples. They used 20 to make lunch and then bought 6 more. How many apples do they have?",
        "a": 9,
    },
    {
        "q": "A library had 48 books. They gave away a third of them and then received 10 new books. How many books does the library have now?",
        "a": 42,
    },
    {
        "q": "Alice has 5 apples. She gives Bob 2 apples, then buys 3 more, and finally gives Carol 1 apple. How many does Alice have?",
        "a": 5,
    },
]

def extract_int(text):
    # Pick the LAST integer mentioned — works well with CoT which ends with the answer.
    nums = re.findall(r"-?\d+", text)
    return int(nums[-1]) if nums else None

In [5]:
def zero_shot_prompt(q):
    return f"Q: {q}\nA:"

def few_shot_prompt(q):
    return (
        "Q: There are 5 cars in a parking lot and 2 more cars arrive. How many cars are in the parking lot?\n"
        "A: 7\n"
        "Q: There are 3 apples. You eat one. How many apples are left?\n"
        "A: 2\n"
        f"Q: {q}\nA:"
    )

def cot_prompt(q):
    return (
        "Q: There are 5 cars in a parking lot and 2 more cars arrive. How many cars are in the parking lot?\n"
        "A: Start with 5 cars. 2 more arrive, so 5 + 2 = 7. Answer: 7\n"
        "Q: Roger has 5 tennis balls. He buys 2 more cans. Each can has 3 balls. How many tennis balls does he have now?\n"
        "A: Roger starts with 5 balls. 2 cans * 3 balls = 6 more balls. 5 + 6 = 11. Answer: 11\n"
        f"Q: {q}\nA: Let's think step by step."
    )

def benchmark(name, prompt_fn, max_new=120):
    hits = 0
    print(f"--- {name} ---")
    for i, item in enumerate(PROBLEMS):
        raw = generate(prompt_fn(item["q"]), max_new_tokens=max_new).strip()
        pred = extract_int(raw)
        ok = "✓" if pred == item["a"] else "✗"
        hits += int(pred == item["a"])
        preview = raw[:80].replace("\n", " ")
        print(f"  {ok} expected={item['a']:<4} got={pred!s:<6} | {preview}")
    print(f"  score: {hits}/{len(PROBLEMS)}\n")
    return hits

score_zero = benchmark("zero-shot",  zero_shot_prompt, max_new=10)
score_few  = benchmark("few-shot",   few_shot_prompt,  max_new=10)
score_cot  = benchmark("CoT",        cot_prompt,       max_new=120)

--- zero-shot ---
  ✗ expected=4    got=None   | Half of the golf balls are blue, so there
  ✗ expected=11   got=6      | He has 5 + 2 = 6 cans of
  ✗ expected=9    got=36     | They had 23 + 20 = 36 apples. They
  ✗ expected=42   got=12     | The library gave away 48 / 3 = 12
  ✗ expected=5    got=10     | Alice has 5 + 2 + 3 = 10 apples
  score: 0/5

--- few-shot ---
  ✗ expected=4    got=None   | Half of the golf balls are blue, so there
  ✗ expected=11   got=6      | He has 5 + 2 = 6 cans of
  ✗ expected=9    got=36     | They used 23 + 20 = 36 apples. They
  ✗ expected=42   got=12     | The library gave away 48 / 3 = 12
  ✗ expected=5    got=10     | Alice has 5 + 2 + 3 = 10 apples
  score: 0/5

--- CoT ---
  ✓ expected=4    got=4      | Half of the golf balls are blue, so there are 16 / 2 = 8 blue golf balls. There 
  ✗ expected=11   got=25     | Roger has 5 + 2 = 6 cans of tennis balls. So, Roger has 6 + 2 = 9 cans of tennis
  ✗ expected=9    got=57     | They used 20 + 6 = 21 a

Depending on the problem, CoT tends to beat zero-shot clearly, and often beats plain few-shot. Note how the CoT completion is much longer — that is *exactly* where CoT spends its tokens: thinking out loud.

## 3. Self-consistency — majority vote over sampled CoTs
Run the same CoT prompt several times at `T > 0` and return the majority answer.

In [6]:
def self_consistency(q, n_samples=5, temperature=0.7):
    votes = []
    for _ in range(n_samples):
        raw = generate(cot_prompt(q), max_new_tokens=120, temperature=temperature)
        pred = extract_int(raw)
        if pred is not None:
            votes.append(pred)
    if not votes:
        return None, votes
    winner, _ = Counter(votes).most_common(1)[0]
    return winner, votes

sc_hits = 0
print("--- self-consistency (5 samples, T=0.7) ---")
for item in PROBLEMS:
    ans, votes = self_consistency(item["q"], n_samples=5)
    ok = "✓" if ans == item["a"] else "✗"
    sc_hits += int(ans == item["a"])
    print(f"  {ok} expected={item['a']:<4} vote={ans!s:<5}  individual votes = {votes}")
print(f"  score: {sc_hits}/{len(PROBLEMS)}")

--- self-consistency (5 samples, T=0.7) ---
  ✗ expected=4    vote=16     individual votes = [16, 6, 4, 16, 9]
  ✗ expected=11   vote=36     individual votes = [12, 19, 36, 36, 28]
  ✗ expected=9    vote=58     individual votes = [58, 59, 20, 24, 33]
  ✗ expected=42   vote=28     individual votes = [24, 28, 37, 63, 28]
  ✗ expected=5    vote=9      individual votes = [9, 9, 1, 10, 11]
  score: 0/5


## 4. When CoT backfires
For a trivial task (e.g., 'What is the first letter of `apple`?'), CoT costs tokens but rarely adds accuracy. The example below contrasts token counts for CoT vs zero-shot on a simple classification.

In [7]:
TRIVIAL = [
    ("What is the first letter of 'apple'?", "a"),
    ("What is the capital of France? Answer in one word.", "paris"),
    ("Translate 'dog' to French. Answer in one word.", "chien"),
]

def trivial_zero(q): return f"Q: {q}\nA:"
def trivial_cot(q):
    return f"Q: {q}\nA: Let's think step by step."

def count_tokens_of(text):
    return len(tokenizer(text, return_tensors="pt").input_ids[0])

print(f"{'question':<55}{'zero-answer':<18}{'zero-tok':>10}{'cot-tok':>10}")
total_zero, total_cot = 0, 0
for q, _ in TRIVIAL:
    ans_zero = generate(trivial_zero(q), max_new_tokens=10).strip()
    ans_cot  = generate(trivial_cot(q),  max_new_tokens=120).strip()
    n_zero, n_cot = count_tokens_of(ans_zero), count_tokens_of(ans_cot)
    total_zero += n_zero; total_cot += n_cot
    print(f"  {q[:50]:<55}{ans_zero[:15]:<18}{n_zero:>10}{n_cot:>10}")
print(f"\n  total tokens — zero-shot = {total_zero},  CoT = {total_cot}")
print("CoT spent", round(total_cot / max(total_zero, 1), 1), "x as many tokens on answers that are already trivial.")

question                                               zero-answer         zero-tok   cot-tok
  What is the first letter of 'apple'?                   a                          3        35
  What is the capital of France? Answer in one word.     london                     5        20
  Translate 'dog' to French. Answer in one word.                                    1        11

  total tokens — zero-shot = 9,  CoT = 66
CoT spent 7.3 x as many tokens on answers that are already trivial.


CoT is a *power tool*, not an 'always on' default — use it where reasoning is actually needed.

## 5. Structured reasoning — `<thinking>` + `<answer>` tags
Have the model write reasoning inside one tag and the final answer inside another, so you can extract only the answer for end users while keeping the reasoning for debugging.

In [8]:
TAGGED_PROMPT = (
    "Write your reasoning inside <thinking>...</thinking> tags, "
    "then the final answer inside <answer>...</answer>.\n\n"
    "Q: {q}\nA:"
)

def parse_tagged(text):
    think = re.search(r"<thinking>(.*?)</thinking>", text, flags=re.S)
    ans   = re.search(r"<answer>(.*?)</answer>",     text, flags=re.S)
    return (think.group(1).strip() if think else None,
            ans.group(1).strip()   if ans   else None)

sample_q = PROBLEMS[1]["q"]
raw = generate(TAGGED_PROMPT.format(q=sample_q), max_new_tokens=150)
reasoning, answer = parse_tagged(raw)
print("raw       :", raw[:200])
print("reasoning :", reasoning)
print("answer    :", answer)

raw       : He has 5 + 2 = 6 cans of tennis balls. So he has 6 * 3 = 18 tennis balls. The final answer: 18.
reasoning : None
answer    : None


A small model like `flan-t5-base` will not always respect the tag format. That is *why* your parser must gracefully return `None` — and why production pipelines retry with a reminder prompt when tags are missing.

## 6. Quick checks

In [9]:
# Integer extractor picks the last integer in a trace
assert extract_int("First 8 balls, then half = 4. Answer: 4") == 4
assert extract_int("no numbers here") is None
# CoT should beat zero-shot by at least 1 question on this suite
assert score_cot >= score_zero, f"CoT ({score_cot}) should not underperform zero-shot ({score_zero})"
# Self-consistency must not underperform a single CoT
assert sc_hits >= max(score_cot - 1, 0), "majority vote should not be much worse than a single CoT"
# On trivial tasks CoT burns more tokens than zero-shot
assert total_cot > total_zero, "CoT should cost more tokens on trivial tasks"
print("OK — in-context learning transfers, CoT helps on reasoning, wastes tokens on trivia.")

OK — in-context learning transfers, CoT helps on reasoning, wastes tokens on trivia.


## Reflection questions

1. The theory says the order of few-shot examples matters because models have recency bias. Design a small experiment to measure that effect on the word-problem suite.
2. Self-consistency needs `T > 0` to produce diversity. If you set `T = 0`, would the majority vote still help? Why / why not?
3. Name two scenarios where CoT *hurts*: once from the theory, and one you can think of yourself.
4. The `<thinking>` + `<answer>` pattern is often paired with a *retry* loop when the tags are missing. Sketch that retry logic in pseudo-code.

## References
- Source theory: [`6-2-few-shot-cot.mdx`](../../llm-quest-theory/level-6/6-2-few-shot-cot.mdx)
- Next: [`6-3-rag-indexing`](6-3-rag-indexing.ipynb)